<a href="https://colab.research.google.com/github/rahilkhan-acadmic/AgenticAI-adventure/blob/main/hello-world.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip3 install uv
!pip3 install python-dotenv
!pip3 install -U langchain-google-genai
!pip3 install -U langchain-openai
!pip install langchain-ollama
!pip install black
!black . --target-version py312
!pip install isort
!isort .

!uv add langchain

All done! ✨ 🍰 ✨
1 file left unchanged.
Skipped 2 files
Resolved 34 packages in 6ms
Checked 33 packages in 2ms


In [2]:
!pip install langchain-groq

In [3]:
from google.colab import userdata
from google import genai

GROK_API_KEY = userdata.get('GROK_API_KEY')
# print("GROK_API_KEY has been loaded.: ",GROK_API_KEY) # Commented out to prevent key leakage

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# print("OPENAI_API_KEY has been loaded.: ",OPENAI_API_KEY) # Commented out to prevent key leakage

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

print("API keys loaded securely.")

API keys loaded securely.


In [8]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
import google.genai as genai

# Ensure API keys are accessible (they are global from previous cells)
# GROK_API_KEY
# GEMINI_API_KEY

class LlmFactory:
  @staticmethod
  def create_llm(model_type: str, model_name: str):
    """
    Factory method to create and return an LLM instance based on type and name.
    """
    if model_type == "groq":
      return ChatGroq(temperature=0, groq_api_key=GROK_API_KEY, model=model_name)
    elif model_type == "gemini":
      genai.configure(api_key=GEMINI_API_KEY)
      return ChatGoogleGenerativeAI(temperature=0, model=model_name, google_api_key=GEMINI_API_KEY)
    else:
      raise ValueError(f"Unsupported model type: {model_type}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [6]:
from groq import Groq

# Initialize the Groq client with your API key
groq_client = Groq(api_key=GROK_API_KEY)

print("Available Groq models:")
for model in groq_client.models.list().data:
    print(f"- {model.id}")

Available Groq models:
- canopylabs/orpheus-v1-english
- openai/gpt-oss-120b
- openai/gpt-oss-safeguard-20b
- llama-3.3-70b-versatile
- whisper-large-v3
- llama-3.1-8b-instant
- canopylabs/orpheus-arabic-saudi
- meta-llama/llama-prompt-guard-2-86m
- groq/compound-mini
- groq/compound
- qwen/qwen3-32b
- meta-llama/llama-4-scout-17b-16e-instruct
- whisper-large-v3-turbo
- meta-llama/llama-prompt-guard-2-22m
- openai/gpt-oss-20b
- allam-2-7b


In [11]:
from re import template
from dotenv import load_dotenv
import os
# The following imports are now handled within LlmFactory or are not directly used here:
# from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
# from langchain_ollama import ChatOllama
# from langchain_groq import ChatGroq
from IPython.display import Markdown
# import google.generativeai as genai # Now configured inside LlmFactory

load_dotenv()

def main():
  print("Hello from langchain-course!")
  information = """
  Iran,[c] officially the Islamic Republic of Iran,[d] also known as Persia,[e] is a country in West Asia.
  It borders Iraq to the west, Turkey, Azerbaijan, and Armenia to the northwest, the Caspian Sea
  to the north, Turkmenistan to the northeast, Afghanistan to the east, Pakistan to the southeast,
  and the Gulf of Oman and the Persian Gulf to the south. With a population of over 92 million,
  Iran ranks 17th globally in both geographic size and population. It is divided into five regions
  with 31 provinces. Capital city Tehran is the nation's largest city and serves as its primary
  economic centre.
  Home to one of the world's oldest continuous major civilizations, the territory of present-day
  Iran was first unified under the Medes in the 7th century BC and reached its territorial
  height in the 6th century BC, when Cyrus the Great founded the Achaemenid Empire.
  Alexander the Great conquered the empire in the 4th century BC.[12] An Iranian rebellion
  in the 3rd century BC established the Parthian Empire, which later liberated the country.
  In the 3rd century AD, the Parthians were succeeded by the Sasanian Empire, which oversaw a
  golden age in the history of Iranian civilization. Ancient Iran saw some of the earliest
  developments of writing, agriculture, urbanization, religion, and administration.
  Once a center for Zoroastrianism, Iran underwent Islamization following the 7th century Muslim
  conquest. Innovations in literature, philosophy, mathematics, medicine, astronomy and art
  were renewed during the Islamic Golden Age and Iranian Intermezzo, a period during which
  Iranian Muslim dynasties ended Arab rule and revived the Persian language. This era was
  followed by Seljuk and Khwarazmian rule, Mongol conquests and the Timurid Renaissance from
  the 11th to 14th centuries.
  """

  summary_template = """
  Given the infromation {information} about a country I want you to create:
  1. A short summary
  2. two interesting facts about them
  3. insteresting facts about their missile technology
  """

  summary_prompt_template = PromptTemplate(
      input_variables =["information"], template = summary_template
  )

  # Use the LlmFactory to create the LLM instance
  model_type = "groq" # You can change this to "gemini" to use Gemini
  llm_model_name = 'llama-3.3-70b-versatile' # Specify the model name for the chosen type

  try:
    llm = LlmFactory.create_llm(model_type, llm_model_name)
    print(f"Successfully created {model_type} LLM with model: {llm_model_name}")
  except ValueError as e:
    print(f"Error creating LLM: {e}")
    return None # Exit if LLM creation fails

  chain = summary_prompt_template | llm

  response = chain.invoke({"information":information})

  if response is None or not hasattr(response, 'content'):
      print("Warning: The AI model did not return a valid response.")
      return None

  # Extract the text content from the response
  if isinstance(response.content, list) and len(response.content) > 0 and isinstance(response.content[0], dict) and 'text' in response.content[0]:
      text_content = response.content[0]['text']
  else:
      text_content = str(response.content) # Fallback to string conversion if format is unexpected

  print(text_content)
  return text_content # Return the extracted text content


if __name__=="__main__":
  # Ensure LlmFactory is defined by executing its cell first
  # This implicit dependency will need the user to run cells in order
  response_text = main()
  if response_text:
      Markdown(response_text)
  else:
      print("No response to display in Markdown.")

Hello from langchain-course!
Successfully created groq LLM with model: llama-3.3-70b-versatile
Based on the provided information, here are the requested points about Iran:

1. **Short Summary**: Iran, also known as Persia, is a country in West Asia with a rich history dating back to the 7th century BC. It has a population of over 92 million, making it the 17th largest country in terms of both geographic size and population. Iran has a diverse history, having been ruled by various empires, including the Achaemenid, Parthian, and Sasanian Empires, and has undergone significant cultural and religious changes, including the adoption of Islam.

2. **Two Interesting Facts**:
   - **Ancient Civilization**: Iran is home to one of the world's oldest continuous major civilizations, with significant contributions to the development of writing, agriculture, urbanization, religion, and administration.
   - **Cultural Revival**: After the 7th century Muslim conquest, Iran experienced a cultural revi